In [ ]:
%load_ext autoreload
%autoreload 2

import pandas as pd
import numpy as np
import os

from dynamicslib.consts import muEM
from dynamicslib.continuation import adaptive_cont
from dynamicslib.common import prop, eom, jacobi_constant
from tqdm.auto import tqdm
from dynamicslib.targeter import dc_overconstrained
from dynamicslib.plotting import plotly_display, test_new_family
from dynamicslib.common_targetters import spatial_perpendicular, fullstate_minus_one, axial, multi_shooter


family = "family_name"
relpath = f"database/{family}.csv"
direction = 1
targetter = spatial_perpendicular(1e-13, muEM)

assert direction in [1,-1]

root = os.getcwd()
file = f"{root}/{relpath}"
db = pd.read_csv(file).set_index("Index")

In [ ]:
keys = ["Initial x", "Initial y", "Initial z", "Initial vx", "Initial vy", "Initial vz"]
keys += ["Period"]
for key in keys:
    if key not in db.columns:
        db[key] = 0.0

im0 = -1 if direction == 1 else 0
im1 = im0 - direction


func = targetter.f_df_stm

# Xm0 = targetter.get_X(prop(db[keys[:-1]].iloc[im0].values,db["Period"].iloc[im0]/2,int_tol=1e-13)[:,-1], db["Period"].iloc[im0])
# Xm1 = targetter.get_X(prop(db[keys[:-1]].iloc[im1].values,db["Period"].iloc[im1]/2,int_tol=1e-13)[:,-1], db["Period"].iloc[im1])
Xm0 = targetter.get_X(db[keys[:-1]].iloc[im0].values, db["Period"].iloc[im0])
Xm1 = targetter.get_X(db[keys[:-1]].iloc[im1].values, db["Period"].iloc[im1])
Xm0, _, _ = dc_overconstrained(Xm0, func, 1e-13, debug=True)
Xm1, _, _ = dc_overconstrained(Xm1, func, 1e-13, debug=True)


f, df, stm = func(Xm0)
svd = np.linalg.svd(df)
tangent = svd.Vh[-1]
if np.dot(tangent, Xm0 - Xm1) < 0:
    tangent *= -1
sprev = np.linalg.norm(Xm0 - Xm1)
print(f"Last dist is {sprev:.3e}")

In [ ]:
Xs, eigs, (DFs, tangents, s_vals) = adaptive_cont(
    Xm0,
    func,
    tangent,
    sprev,
    1e-5,
    5.5,
    5e-12,
    max_iter=9,
    target_iter=4,
    rate=1.15,
    reduce_maxiter=1.05,
    exact_tangent=True,
    exp_iters=0.7,
    pseudo=False,
)

In [ ]:
param_names = ["Index"] + keys + ["Jacobi Constant", "Stability Index"]
param_names += ["Eig1", "Eig2", "Eig3", "Eig4", "Eig5", "Eig6"]

xyzs = []
data = []


iterator = list(enumerate(zip(Xs, eigs)))[::direction]
for i, (X, evals) in tqdm(iterator):
    Xcp = X.copy()
    x0 = targetter.get_x0(Xcp)
    tf = targetter.get_period(Xcp)
    xyzs.append(prop(x0, tf, muEM, 1e-10))
    args = np.argsort(np.abs(evals))
    stabind = np.max([(np.abs(lam) + 1 / np.abs(lam)) / 2 for lam in evals])
    jc = jacobi_constant(x0)
    x0 = targetter.get_x0(Xcp)
    data.append([i, *x0, tf, jc, stabind, *evals])
if direction == -1 and len(xyzs):
    xyzs = xyzs[::-1]

In [ ]:
df = pd.DataFrame(data, columns=param_names).set_index("Index")
ind0 = db.index[0]
db = db.reset_index(drop=True)
print(ind0)

if direction == -1:
    db.index += len(df)
    df.index = df.index[::-1]
    full = pd.concat((df, db))
    if ind0 < 0:
        full.index += ind0 - len(df)
else:
    df.index += len(db)
    full = pd.concat((db, df))
    full.index += ind0

for key in keys:
    if np.all(np.logical_or(full[key].abs() < 1e-8, np.isnan(full[key]))):
        full = full.drop(columns=[key])
full = full.sort_index()
full.index.name = "Index"

full.to_csv(family + ".csv")

In [ ]:
test_new_family(full, "L2 Halo")

In [ ]:
a = -20
b = None
c = 1
# colscale = [
#     [0.0, "rgb(255, 0, 0)"],
#     [0.2, "rgb(255, 10, 0)"],
#     [0.35, "rgb(200, 200, 30)"],
#     [0.75, "rgb(200, 85, 15)"],
#     [1.0, "rgb(255, 0, 0)"],
# ]
plotly_display(xyzs[a:b:c], df[a:b:c], port=8004, color_by="Index")